In [4]:
from __future__ import annotations
from typing import Iterable, Callable

import random
import numpy as np
import pandas as pd
from math import log
import networkx as nx
from scipy.stats import pearsonr
from itertools import combinations
from sklearn.metrics import adjusted_rand_score

from freyrelab.regnets import regnet
from freyrelab.regnets.abasy import Abasy
from freyrelab.regnets.regnet import RegNet
from freyrelab.nets import models, dissimilarity

from netective.utils import get_clusters
from netective.structure.structure import association, compare_structure, create_symmetric_heatmap


In [2]:
abasy = Abasy(db='abasy_internal', expire_cache=True)
regnet_ids = abasy.select_regnets(nr_strong_wa=True)    # select all regnets without redundancy, keep the strong one if available
regnets = abasy.regnet(regnet_ids)                      # get the regnets {regnet_id: RegNet}


seed = 42
random.seed(seed)
random_graph = {}
hm_seed_size = 3

for net_id, G in regnets.items():
    n = G.number_of_nodes()
    m = G.number_of_edges()
    random_graph[f'BA_{net_id}'] = models.barabasi_albert_graph(n)
    random_graph[f'SF_{net_id}'] = nx.scale_free_graph(n, seed=seed)
    random_graph[f'ER_{net_id}'] = nx.erdos_renyi_graph(n, m/(n*(n-1)), seed=seed, directed=False)
    random_graph[f'HM_{net_id}'] = models.hierarchical_modular_graph(int(log(n, hm_seed_size))-1, m=hm_seed_size)

networks = {**regnets, **random_graph}
# networks = {name: nx.Graph(G) for name, G in networks.items()}
networks = {name: nx.DiGraph(G) for name, G in networks.items()}

In [7]:
networks_nodata = {name: nx.DiGraph([(n1, n2) for n1, n2, d in graph.edges(data=True)]) for name, graph in networks.items()}

my_props = ['Complex Feed-Forward Circuits',
 'Density',
 'Entropy of Out-Degree Distribution',
 'Feed-Forward Circuits',
 '3-Feedback Loops',
 'Gini Index',
 'Max In-Dregree',
 'Max Out-Degree',
 'Number of Arcs',
 'Regulators',
 'Self Regulations']


scalar, dists = compare_structure(networks=networks_nodata, norm='network', return_prop_dicts=True, workers=6)#, selected_props = my_props)
df = association(scalar, corr_func=pearsonr)
# fig_scalar, _ = create_symmetric_heatmap(df, title=f"Global properties")
# index = list(corr_df.index)
models = {
    'BA' : 0,
    'SF' : 1,
    'ER' : 2,
    'HM' : 3,
    }

gs = [models.get(name.split('_')[0], 4) for name in df.index] # 4 for biological
result = get_clusters(df, 5, map_ids=False) # 5 clusters

adjusted_rand_score(gs, result)


Properties used for analysis: 
Average Degree for Nearest Neighbors (Undirected)
Average Local Efficiency
Average Degree for Nearest Neighbors (Out-Out)
Average Shortest Path Length
Betweenness Centrality
Center
Clustering Coefficient
Complex Feed-Forward Circuits
Degree
Density
Diameter
Eccentricity
Entropy of Degree Distribution
Entropy of Out-Degree Distribution
Feed-Forward Circuits
3-Feedback Loops
Gene % in the Giant Component
Gini Index
Global Efficiency
In-Degree
Locality Index
Max Degree
Max In-Dregree
Max Out-Degree
Number of Arcs
Number of Edges
Number of Nodes
Out-Degree
Periphery
Radius
Regulators
Rich Club Coefficient
Self-Loops
Self Regulations
Subgraph Centrality
Undirected Density
Undirected Gini Index




  0%|          | 0/7 [00:00<?, ?it/s]

Running 100226_v2019_sA22-DBSCR15_eStrong
Running 158878_v2015_sRTB13
Running 158879_v2015_sRTB13
Running 160491_v2015_sRTB13
Running 186103_v2015_sRTB13
Running 196620_v2015_sRTB13
Running 196627_v2020_s21_eStrong
Running 198466_v2015_sRTB13
Running 199310_v2015_sRTB13
Running 208964_v2020_sRPA20_eStrong
Running 224308_v2008_sDBTBS08_eStrong
Running 273036_v2015_sRTB13
Running 282458_v2015_sRTB13
Running 282459_v2015_sRTB13
Running 301447_v2015_sRTB13
Running 316385_v2015_sRTB13
Running 319701_v2015_sRTB13
Running 331111_v2015_sRTB13
Running 331112_v2015_sRTB13
Running 340184_v2015_sRTB13
Running 344601_v2015_sRTB13
Running 359786_v2015_sRTB13
Running 359787_v2015_sRTB13
Running 362663_v2015_sRTB13
Running 364106_v2015_sRTB13
Running 367830_v2015_sRTB13
Running 370551_v2015_sRTB13
Running 370552_v2015_sRTB13
Running 370554_v2015_sRTB13
Running 381754_v2015_sRTB13
Running 405955_v2015_sRTB13
Running 406558_v2015_sRTB13
Running 418127_v2015_sRTB13
Running 426430_v2015_sRTB13
Running 439

210it [01:53,  1.85it/s]                     


1.0

In [4]:
result = get_clusters(df, 5, map_ids=True) # 5 clusters
result

{2: ['160491_v2015_sRTB13',
  '186103_v2015_sRTB13',
  '198466_v2015_sRTB13',
  '100226_v2019_sA22-DBSCR15_eStrong',
  '301447_v2015_sRTB13',
  '319701_v2015_sRTB13',
  '208964_v2020_sRPA20_eStrong',
  '370551_v2015_sRTB13',
  '370554_v2015_sRTB13',
  '370552_v2015_sRTB13',
  '381754_v2015_sRTB13',
  '406558_v2015_sRTB13',
  '487214_v2015_sRTB13',
  '83332_v2018_s11-12-15-16'],
 5: ['158878_v2015_sRTB13',
  '196620_v2015_sRTB13',
  '158879_v2015_sRTB13',
  '199310_v2015_sRTB13',
  '282458_v2015_sRTB13',
  '273036_v2015_sRTB13',
  '282459_v2015_sRTB13',
  '316385_v2015_sRTB13',
  '331111_v2015_sRTB13',
  '331112_v2015_sRTB13',
  '344601_v2015_sRTB13',
  '340184_v2015_sRTB13',
  '362663_v2015_sRTB13',
  '359787_v2015_sRTB13',
  '359786_v2015_sRTB13',
  '364106_v2015_sRTB13',
  '367830_v2015_sRTB13',
  '405955_v2015_sRTB13',
  '439855_v2015_sRTB13',
  '418127_v2015_sRTB13',
  '426430_v2015_sRTB13',
  '478008_v2015_sRTB13',
  '481805_v2015_sRTB13',
  '93062_v2015_sRTB13',
  '451516_v2015_s

In [5]:
scalar

{'160491_v2015_sRTB13': {'Average Local Efficiency': 0.03201001786074381,
  'Average Shortest Path Length': 0.0300927475480069,
  'Center': 0.006578947368421052,
  'Diameter': 0.06622516556291391,
  'Global Efficiency': 0.2762883339695818,
  'Periphery': 0.18421052631578946,
  'Radius': 0.033112582781456956,
  'Complex Feed-Forward Circuits': 0.0,
  'Feed-Forward Circuits': 0.0,
  '3-Feedback Loops': 5.599955200358397e-06,
  'Entropy of Degree Distribution': 0.22640547677957343,
  'Max Degree': 0.18604651162790697,
  'Self-Loops': 0.03488372093023256,
  'Undirected Gini Index': 0.5323501173458502,
  'Density': 0.01601454293628809,
  'Entropy of Out-Degree Distribution': 0.2236577066281137,
  'Gini Index': 0.5265278528934558,
  'Max In-Dregree': 0.18023255813953487,
  'Max Out-Degree': 0.18023255813953487,
  'Regulators': 1.0,
  'Self Regulations': 0.03488372093023256,
  'Gene % in the Giant Component': 1.0,
  'Undirected Density': 0.008137119113573408,
  'Average Average Degree for Nea

In [8]:
len(['Average Degree for Nearest Neighbors (Undirected)',
 'Average Local Efficiency',
 'Average Degree for Nearest Neighbors (Out-Out)',
 'Average Shortest Path Length',
 'Betweenness Centrality',
 'Center',
 'Clustering Coefficient',
 'Complex Feed-Forward Circuits',
 'Degree',
 'Density',
 'Diameter',
 'Eccentricity',
 'Entropy of Degree Distribution',
 'Entropy of Out-Degree Distribution',
 'Feed-Forward Circuits',
 '3-Feedback Loops',
 'Gene % in the Giant Component',
 'Gini Index',
 'Global Efficiency',
 'In-Degree',
 'Locality Index',
 'Max Degree',
 'Max In-Dregree',
 'Max Out-Degree',
 'Number of Arcs',
 'Number of Edges',
 'Number of Nodes',
 'Out-Degree',
 'Periphery',
 'Radius',
 'Regulators',
 'Rich Club Coefficient',
 'Self-Loops',
 'Self Regulations',
 'Subgraph Centrality',
 'Undirected Density',
 'Undirected Gini Index']) + 3*11

70

In [13]:
from math import factorial as f
total = 0
n = 11

for r in range(n):
    x = f(n) / f(r) / f(n-r)
    total += x

(total*60)/3600

34.11666666666667

In [14]:
(210*209)/2

21945.0

In [22]:
import inspect
from netective.structure import properties
parent_class = properties._Property

directed_scalar_properties = []
for name, obj in inspect.getmembers(properties):
        if inspect.isclass(obj) and issubclass(obj, parent_class) and obj != parent_class:
            if obj._use_direction and obj._return_type == 'scalar':
                directed_scalar_properties.append(obj.CLASS_NAME)

directed_scalar_properties

['Complex Feed-Forward Circuits',
 'Density',
 'Entropy of Out-Degree Distribution',
 'Feed-Forward Circuits',
 '3-Feedback Loops',
 'Gini Index',
 'Max In-Dregree',
 'Max Out-Degree',
 'Number of Arcs',
 'Regulators',
 'Self Regulations']

In [8]:
networks_nodata = {name: nx.DiGraph([(n1, n2) for n1, n2, d in graph.edges(data=True)]) for name, graph in networks.items()}


scalar, dists = compare_structure(networks=networks_nodata, norm='network', return_prop_dicts=True, workers=6, selected_props=directed_scalar_properties)
df = association(scalar, corr_func=pearsonr)
fig_scalar, _ = create_symmetric_heatmap(df, title=f"Global properties")
# index = list(corr_df.index)
models = {
    'BA' : 0,
    'SF' : 1,
    'ER' : 2,
    'HM' : 3,
    }

gs = [models.get(name.split('_')[0], 4) for name in df.index] # 4 for biological
result = get_clusters(df, 5, map_ids=False) # 5 clusters

adjusted_rand_score(gs, result)


Properties used for analysis: 
Average Local Efficiency
Average Shortest Path Length
Center
Complex Feed-Forward Circuits
Density
Diameter
Entropy of Degree Distribution
Entropy of Out-Degree Distribution
Feed-Forward Circuits
3-Feedback Loops
Gene % in the Giant Component
Gini Index
Global Efficiency
Max Degree
Max In-Dregree
Max Out-Degree
Number of Arcs
Number of Edges
Number of Nodes
Periphery
Radius
Regulators
Self-Loops
Self Regulations




  0%|          | 0/7 [00:00<?, ?it/s]

Running 100226_v2019_sA22-DBSCR15_eStrong
Running 158878_v2015_sRTB13
Running 158879_v2015_sRTB13
Running 160491_v2015_sRTB13
Running 186103_v2015_sRTB13
Running 196620_v2015_sRTB13
Running 196627_v2020_s21_eStrong
Running 198466_v2015_sRTB13
Running 199310_v2015_sRTB13
Running 208964_v2020_sRPA20_eStrong
Running 224308_v2008_sDBTBS08_eStrong
Running 273036_v2015_sRTB13
Running 282458_v2015_sRTB13
Running 282459_v2015_sRTB13
Running 301447_v2015_sRTB13
Running 316385_v2015_sRTB13
Running 319701_v2015_sRTB13
Running 331111_v2015_sRTB13
Running 331112_v2015_sRTB13
Running 340184_v2015_sRTB13
Running 344601_v2015_sRTB13
Running 359786_v2015_sRTB13
Running 359787_v2015_sRTB13
Running 362663_v2015_sRTB13
Running 364106_v2015_sRTB13
Running 367830_v2015_sRTB13
Running 370551_v2015_sRTB13
Running 370552_v2015_sRTB13
Running 370554_v2015_sRTB13
Running 381754_v2015_sRTB13
Running 405955_v2015_sRTB13
Running 406558_v2015_sRTB13
Running 418127_v2015_sRTB13
Running 426430_v2015_sRTB13
Running 439

75it [00:11,  2.91it/s]                      

undirected, solo scalars + promedios: 0.6661316415704341
undirected, solo scalars: 0.6824370232071961

DiGraph(G), solo scalars: 1          But why...............
DiGraph(G), 4 momentos: 0.3622903352329096
DiGraph(G), + promedios: 1          Pero en el heatmap se nota menos. 